## Proyecto Práctico – Procesamiento de un Dataset con Spark utilizando DataFrames y SparkSQL
###### Contexto: El REMS o “Rover Environmental Monitoring Station” es una estación meteorológica en Marte creada para el rover Curiosity y provista por España y Finlandia. El REMS mide la humedad, la presión, la temperatura, la velocidad del viento y la radiación ultravioleta en Marte. Este proyecto español está liderado por el Centro Español de Astrobiología e incluye como socio al Instituto Meteorológico finlandés, aportando sensores de presión y humedad.

###### Dataset: El dataset contiene 10 columnas con los siguientes encabezados en orden de aparicion: earth_date_time,	sol_number,	max_ground_temp(°C),	min_ground_temp(°C),	max_air_temp(°C),	min_air_temp(°C),	mean_pressure(Pa),	sunrise	sunset,	UV_Radiation.
###### Objetivo: A partir del dataset con las observaciones del clima del planeta Marte recabados por el Mars Curiosity Rover, obtener los valores para las siguientes métricas:
1.	Temperatura máxima del suelo del planeta y día de la medición.
2.	Temperatura mínima del aire del planeta y día de la medición.
3.	Presión atmosférica promedio máxima obtenida y el valor del delta de temperatura del aire pare ese mismo día de la medición.
4.  Filtrar las observaciones con Radiación UV con la categoria "very_high" e indicar los valores de temperatura máxima registrada.

# 1. Configuración de PySpark en Google Colab

In [1]:
# Instalar Java
!apt-get install openjdk-8-jdk-headless -qq > /dev/null

# Descargar Spark 3.5.1
!wget -q https://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz

# Extraer los archivos de Spark
!tar xf spark-3.5.1-bin-hadoop3.tgz

# Instalar findspark para facilitar el uso de Spark en Python
!pip install -q findspark

# Configurar las variables de entorno para Java y Spark
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"

# Inicializar Spark usando findspark
import findspark
findspark.init()

# Crear la sesión de Spark
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()

# Verificar la versión de Spark
print(f"Versión de Spark: {spark.version}")



Versión de Spark: 3.3.2


# 2. Montamos Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
input_path = '/content/drive/MyDrive/BDPS - Notebooks/Ficheros Input Notebooks/Clase 1/REMS_Mars_Dataset.csv'

# Leer el archivo con textFile
lines = spark.sparkContext.textFile(input_path)

# Leer una vista previa del archivo CSV con samplingRatio
df_preview = spark.read.format("csv") \
    .option("samplingRatio", 0.01) \
    .option("inferSchema", "true") \
    .option("header", "false") \
    .load(input_path)

# Mostrar el DataFrame
df_preview.show()

# 3. Importar los tipos de datos requeridos para construir el esquema

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, FloatType, TimestampType
from pyspark.sql.functions import col, expr, year, concat_ws, to_timestamp, round, array_contains, substring, to_date

# 4. Crear el esquema para el DataFrame

In [ ]:
schema = StructType([
    StructField("FechaTierra", StringType(), True),
    StructField("NumeroDeSol", StringType(), True),
    StructField("MaxTempSueloCent", FloatType(), True),
    StructField("MinTempSueloCent", FloatType(), True),
    StructField("MaxTempAireCent", FloatType(), True),
    StructField("MinTempAireCent", FloatType(), True),
    StructField("PresAtmMediaPas", FloatType(), True),
    StructField("AmanecerHora", StringType(), True),
    StructField("AtardecerHora", StringType(), True),
    StructField("RadiacionUV", StringType(), True),
])

# 5. Crear el DataFrame e imprimir el esquema

In [ ]:
# Definir la ruta del archivo en Google Drive
input_path = '/content/drive/MyDrive/BDPS - Notebooks/Ficheros Input Notebooks/Clase 1/REMS_Mars_Dataset.csv'

# Leer el archivo CSV con un esquema definido
df_mars_weater = spark.read.format("csv") \
    .option("header", "false") \
    .schema(schema) \
    .load(input_path)

# Imprimir el esquema del DataFrame
df_mars_weater.printSchema()


# 6. Listar las columnas del DataFrame

In [ ]:
df_mars_weater.columns

# 7. Listar las 10 primeras filas sin truncar el resultado

In [ ]:
df_mars_weater.show(100, truncate=False)

# 8. Crear una columna que contenga la diferencia (delta) de temperatura del aire

In [ ]:
df_mars_diff_temp = df_mars_weater.select(col("FechaTierra"), col("MaxTempAireCent"), col("MinTempAireCent"), expr("MaxTempAireCent - MinTempAireCent").alias("DiffTempAireCent"))
df_mars_diff_temp.show(10, truncate=False)

# 9. Filtrar por condiciones de temperatura

In [ ]:
df_mars_diff_temp.filter((col("MaxTempAireCent") >= 8) & (col("DiffTempAireCent") > 95)).show(10, truncate=False)

In [ ]:
df_mars_diff_temp.where("MaxTempAireCent >= 8 and DiffTempAireCent > 95").show(10, truncate=False)

# 10. Calcular la diferencia en horas, minutos y segundos entre el amanecer y el atardecer en el planeta.

In [ ]:
df_mars_hours_day = df_mars_weater.withColumn("Amanecer_timestamp", to_timestamp(col("AmanecerHora")))\
                                  .withColumn("Atardecer_timestamp", to_timestamp(col("AtardecerHora")))\
                                  .withColumn('DiffInSeconds',col("Atardecer_timestamp").cast("long") - col('Amanecer_timestamp').cast("long"))\
                                  .withColumn('DiffInMinutes',round(col('DiffInSeconds')/60))\
                                  .withColumn('DiffInHours',round(col('DiffInMinutes')/60))

df_mars_hours_day.select(col("FechaTierra"), col("AmanecerHora"), col("AtardecerHora"), col('DiffInHours').alias("Dif@Horas"), col('DiffInMinutes').alias("Dif@Minutos"), col('DiffInSeconds').alias("Dif@Segundos")).show(15, truncate=False)

# 11. Crear columnas para los componentes de la fecha y transformar la columa "FechaTierra" al tipo Date en una nueva columna con el nombre "FechaObservacion"

In [ ]:
df_mars_earth_time =  df_mars_hours_day.withColumn("Año_Tierra", substring("FechaTierra", 8, 4))\
                                       .withColumn("Mes_Tierra", substring("FechaTierra",13, 2))\
                                       .withColumn("Dia_Tierra", substring("FechaTierra", 16, 2))\
                                       .withColumn("FechaObservacion", to_date(substring("FechaTierra", 8, 10)))
df_mars_earth_time.select("FechaTierra","Año_Tierra","Mes_Tierra","Dia_Tierra","FechaObservacion").orderBy(col("FechaObservacion").asc()).show(10, truncate=False)

# 12. Temperatura máxima del suelo del planeta y día de la medición.

In [ ]:
df_mars_earth_time.select(col("FechaObservacion"), col("MaxTempSueloCent")).orderBy(col("MaxTempSueloCent").desc()).show(10, truncate=False)


# 13. Temperatura mínima del aire del planeta y día de la medición.

In [ ]:
df_mars_earth_time.select(col("FechaObservacion"), col("MinTempAireCent")).orderBy(col("MinTempAireCent").asc()).show(10, truncate=False)

# 14. Presión atmosférica promedio máxima obtenida y el valor del delta de temperatura del aire pare ese mismo día de la medición.

In [ ]:
df_mars_earth_time.select(col("FechaObservacion"), col("PresAtmMediaPas"), (col("MaxTempAireCent") - col("MinTempAireCent"))\
                          .alias("DeltaTempAireCent")).orderBy(col("PresAtmMediaPas").desc()).show(10, truncate=False)

In [ ]:
df_mars_earth_time.select(col("FechaObservacion"), col("PresAtmMediaPas"), (col("MaxTempAireCent") - col("MinTempAireCent"))\
                          .alias("DeltaTempAireCent")).orderBy(col("PresAtmMediaPas").desc(), col("DeltaTempAireCent").asc()).show(10, truncate=False)

# 15. Filtrar las observaciones con Radiación UV con la categoria "very_high" e indicar los valores de temperatura máxima registrada.

In [ ]:
df_mars_earth_time.filter(col("RadiacionUV").contains("very_high")).select(col("FechaObservacion"), col("MaxTempAireCent"), col("RadiacionUV")).orderBy(col("MaxTempAireCent").desc()).show(10)

In [ ]:
df_mars_earth_time.createOrReplaceTempView("MARSWEATHER")
spark.sql("SELECT FechaObservacion, MaxTempAireCent, RadiacionUV FROM MARSWEATHER WHERE RadiacionUV LIKE '%very_high%' ORDER BY MaxTempAireCent DESC ").show(10)